# features

> Waveform preparation and Kaldi log-mel for the Audio-JEPA encoder.
>
> The frontend is `torchaudio.compliance.kaldi.fbank` as the training repo
> calls it: 32 kHz mono, global mean removed, Hanning window, 128 mels,
> then pad or trim to 256 frames. Short clips are tiled so a patch is full of sound, and pooling skips silent time rows.


In [ ]:
#| default_exp features


In [ ]:
#| hide
from nbdev.showdoc import *


In [ ]:
#| export
from __future__ import annotations


In [ ]:
#| export
import wave
from pathlib import Path

import numpy as np

from nada.config import AudioJEPAConfig


In [ ]:
#| export
def read_wav(path: str | Path, sample_rate: int = 32_000) -> np.ndarray:
    """Read a 16-bit PCM WAV file as mono float32 in `[-1, 1]`.

    Resamples with linear interpolation when the file is not at `sample_rate`.
    """
    with wave.open(str(path), "rb") as handle:
        rate, width, channels = handle.getframerate(), handle.getsampwidth(), handle.getnchannels()
        frames = handle.readframes(handle.getnframes())
    if width != 2:
        raise ValueError(f"{path}: expected 16-bit PCM, got {8 * width}-bit")
    samples = np.frombuffer(frames, dtype="<i2").astype(np.float32) / 32768.0
    mono = samples.reshape(-1, channels).mean(axis=1)
    return resample(mono, rate, sample_rate) if rate != sample_rate else mono


In [ ]:
#| export
def resample(waveform: np.ndarray, source_rate: int, dest_rate: int) -> np.ndarray:
    """Linear-interpolate `waveform` from `source_rate` to `dest_rate`."""
    wave = np.asarray(waveform, dtype=np.float32).reshape(-1)
    if source_rate == dest_rate or wave.size == 0:
        return wave
    length = max(int(round(wave.size * dest_rate / source_rate)), 1)
    source = np.linspace(0.0, 1.0, wave.size, endpoint=False)
    dest = np.linspace(0.0, 1.0, length, endpoint=False)
    return np.interp(dest, source, wave).astype(np.float32)


In [ ]:
#| export
def center(waveform: np.ndarray) -> np.ndarray:
    """Subtract the clip mean, as the training transform does before fbank."""
    wave = np.asarray(waveform, dtype=np.float32).reshape(-1)
    return wave - wave.mean()


In [ ]:
#| export
def _next_power_of_2(n: int) -> int:
    return 1 if n <= 0 else 1 << (n - 1).bit_length()


def _mel(freq: np.ndarray | float):
    return 1127.0 * np.log(1.0 + np.asarray(freq, dtype=np.float64) / 700.0)


def mel_banks(num_bins: int, padded_window: int, sample_rate: float, low_freq: float, high_freq: float) -> np.ndarray:
    """Kaldi triangular mel filterbank of shape `(num_bins, padded_window // 2 + 1)`."""
    num_fft = padded_window // 2
    fft_bin_width = sample_rate / padded_window
    mel_low, mel_high = _mel(low_freq), _mel(high_freq)
    delta = (mel_high - mel_low) / (num_bins + 1)
    bins = np.arange(num_bins, dtype=np.float64)[:, None]
    left, center, right = mel_low + bins * delta, mel_low + (bins + 1) * delta, mel_low + (bins + 2) * delta
    mel = _mel(fft_bin_width * np.arange(num_fft, dtype=np.float64))[None]
    slopes = np.maximum(0.0, np.minimum((mel - left) / (center - left), (right - mel) / (right - center)))
    return np.pad(slopes, ((0, 0), (0, 1))).astype(np.float64)


In [ ]:
#| export
def fbank(waveform: np.ndarray, config: AudioJEPAConfig | None = None) -> np.ndarray:
    """Kaldi log-mel of a mono clip, matching `torchaudio.compliance.kaldi.fbank`."""
    config = config or AudioJEPAConfig()
    wave = center(waveform)
    window_shift = int(config.sample_rate * config.hop_length_ms * 0.001)
    window_size = int(config.sample_rate * config.frame_length_ms * 0.001)
    padded = _next_power_of_2(window_size)
    if wave.size < window_size:
        wave = np.pad(wave, (0, window_size - wave.size))
    frames = 1 + (wave.size - window_size) // window_shift
    strided = np.lib.stride_tricks.as_strided(
        wave, shape=(frames, window_size), strides=(wave.strides[0] * window_shift, wave.strides[0])
    ).copy()
    strided = strided - strided.mean(axis=1, keepdims=True)
    offset = np.pad(strided, ((0, 0), (1, 0)), mode="edge")[:, :-1]
    strided = strided - 0.97 * offset
    n = np.arange(window_size, dtype=np.float64)
    window = 0.5 - 0.5 * np.cos(2 * np.pi * n / (window_size - 1))
    strided = np.pad(strided * window, ((0, 0), (0, padded - window_size)))
    spectrum = np.abs(np.fft.rfft(strided, n=padded)) ** 2
    banks = mel_banks(config.n_mels, padded, config.sample_rate, config.f_min, config.f_max)
    energy = np.maximum(spectrum @ banks.T, np.finfo(np.float32).eps)
    return np.log(energy).astype(np.float32)


In [ ]:
#| export
def mel_spec(waveform: np.ndarray, config: AudioJEPAConfig | None = None) -> np.ndarray:
    """Log-mel padded or trimmed to `config.time_bins`."""
    config = config or AudioJEPAConfig()
    spec = fbank(waveform, config)
    if spec.shape[0] < config.time_bins:
        spec = np.pad(spec, ((0, config.time_bins - spec.shape[0]), (0, 0)))
    else:
        spec = spec[: config.time_bins]
    return spec


In [ ]:
#| export
def prepare(waveforms: list[np.ndarray] | np.ndarray, config: AudioJEPAConfig | None = None) -> np.ndarray:
    """Stack clips as an NHWC spectrogram `(batch, time, mel, 1)`."""
    config = config or AudioJEPAConfig()
    if isinstance(waveforms, np.ndarray) and waveforms.ndim == 1:
        waveforms = [waveforms]
    specs = [mel_spec(wave, config) for wave in waveforms]
    return np.stack(specs, axis=0)[..., None].astype(np.float32)


In [ ]:
#| export
def tile_short(waveform: np.ndarray, config: AudioJEPAConfig | None = None, min_patches: int | None = None) -> np.ndarray:
    """Repeat a short clip until it fills `min_patches` time rows (~625 ms each).

    The default fills the whole 10 s canvas. A click then occupies every patch
    instead of one row plus silence, which is what the encoder was trained on.
    Clips already that long are left alone.
    """
    config = config or AudioJEPAConfig()
    wave = np.asarray(waveform, dtype=np.float32).reshape(-1)
    if min_patches is None:
        min_patches = config.grid_time
    patch_samples = max(int(round(config.clip_samples / config.grid_time)), 1)
    need = min(max(min_patches, 1) * patch_samples, config.clip_samples)
    if wave.size == 0 or wave.size >= need:
        return wave[: config.clip_samples]
    return np.tile(wave, int(np.ceil(need / wave.size)))[:need]

In [ ]:
#| export
def row_energy(spectrogram: np.ndarray, config: AudioJEPAConfig | None = None) -> np.ndarray:
    """Mean |log-mel| in each time-patch row. Shape `(batch, grid_time)`."""
    config = config or AudioJEPAConfig()
    spec = np.asarray(spectrogram, dtype=np.float32)
    if spec.ndim == 2:
        spec = spec[None, ..., None]
    elif spec.ndim == 3:
        spec = spec[..., None]
    batch = spec.shape[0]
    rows = spec.reshape(batch, config.grid_time, config.patch_h, config.grid_freq, config.patch_w)
    return np.mean(np.abs(rows), axis=(2, 3, 4))


def pool_tokens(tokens: np.ndarray, spectrogram: np.ndarray, config: AudioJEPAConfig | None = None) -> np.ndarray:
    """Mean of tokens whose time row has spectrogram energy. Shape `(batch, dim)`."""
    config = config or AudioJEPAConfig()
    tokens = np.asarray(tokens, dtype=np.float32)
    energy = row_energy(spectrogram, config)
    floor = np.maximum(energy.max(axis=1, keepdims=True) * 0.05, 1e-4)
    active = energy >= floor
    weights = np.repeat(active, config.grid_freq, axis=1)[..., None].astype(np.float32)
    counted = weights.sum(axis=1)
    pooled = (tokens * weights).sum(axis=1) / np.maximum(counted, 1.0)
    empty = counted.reshape(-1) < 1
    if empty.any():
        pooled[empty] = tokens[empty].mean(axis=1)
    return pooled

In [ ]:
from nada.config import tiny_config

cfg = tiny_config()
clip = np.random.default_rng(0).standard_normal(cfg.sample_rate).astype(np.float32)
spec = mel_spec(clip, cfg)
assert spec.shape == (cfg.time_bins, cfg.n_mels)
batch = prepare([clip, clip[: cfg.sample_rate // 2]], cfg)
assert batch.shape == (2, cfg.time_bins, cfg.n_mels, 1)

short = clip[: cfg.sample_rate // 20]
filled = tile_short(short, cfg, min_patches=2)
assert filled.size == 2 * (cfg.clip_samples // cfg.grid_time)
assert np.allclose(tile_short(clip, cfg), clip)

tokens = np.arange(2 * cfg.num_patches * 4, dtype=np.float32).reshape(2, cfg.num_patches, 4)
energy = np.zeros((2, cfg.time_bins, cfg.n_mels, 1), dtype=np.float32)
energy[:, : cfg.patch_h] = 1.0
pooled = pool_tokens(tokens, energy, cfg)
assert pooled.shape == (2, 4)
assert np.allclose(pooled[0], tokens[0, : cfg.grid_freq].mean(0))
